# Feature Track 4: Tools and Agents

A **tool** is a function the LLM can choose to call when it cannot answer from its own knowledge. The LLM does not execute the tool, it outputs a structured request, and the application runs the function and feeds the result back.

This track builds up to a full RAG agent step by step:

| Notebook | What it adds |
|---|---|
| **4a** (this one) | Define a tool; run the request-execute-respond loop manually |
| **4b** | Wrap the loop in a `ToolAgent`; multi-tool agents |
| **4c** | RAG retriever exposed as a tool; manual loop |
| **4d** | RAG retriever tool inside a `ToolAgent` |
| **4e** | Full RAG agent wrapped as a tool for a parent agent (subagent pattern) |

In [ ]:
import json

from conversational_toolkit.llms.base import LLMMessage, Roles, MessageContent
from conversational_toolkit.llms.openai import OpenAILLM

from sme_kt_zh_collaboration_rag.feature4_tool_agents import SumTwoNumbers

---

## Define a Tool

Every tool exposes three things to the LLM: a **name**, a **description** (natural language, tells the LLM when to use it), and a **JSON Schema** for its parameters (tells the LLM what to pass). The `call` method contains the actual logic.

Raising errors inside `call` is intentional. The LLM receives the error message and can retry with corrected arguments.

```python
class Tool(ABC):
    name: str
    description: str
    parameters: dict[str, Any]

    @abstractmethod
    async def call(self, args: dict[str, Any]) -> dict[str, Any]: ...

    def json_schema(self) -> ToolDescription:
        return {"type": "function", "function": {"name": self.name, "description": self.description, "parameters": self.parameters}}
```

In [ ]:
# Create an instance of the tool with appropriate metadata to describe its functionality
tool_sum = SumTwoNumbers(
    name="sum_two_numbers",  # Name, how it can be called
    description="A tool to sum two numbers. It takes two numbers as input and returns their sum.",  # What it does
    # What parameters it expects
    parameters={
        "type": "object",
        "properties": {
            # The first parameter
            "number_1": {
                "type": "number",  # Type of the parameter
                "description": "The first number to be summed.",  # Description of the parameter
            },
            # The second parameter
            "number_2": {
                "type": "number",
                "description": "The second number to be summed.",
            },
        },
        "required": ["number_1", "number_2"],  # Which parameters are required
        "additionalProperties": False,  # No additional properties allowed
    },
)

print(await tool_sum.call({"number_1": 5, "number_2": 10}))

---

## LLM Using a Tool

When given a tool-enabled LLM, the call-execute-respond cycle has three steps:

1. **Request**: send the user message; the LLM responds with a tool call request (no answer yet)
2. **Execute**: run the requested tool(s) and collect the results
3. **Respond**: send the results back to the LLM; it generates the final answer

`tool_choice="auto"` lets the LLM decide whether to use a tool, it will answer directly if it can.

## Connect LLM to Tool

Once the tool defined in a format understandable for the LLM, and that can be used to constraint it's usage, it can be provided to the LLM.

In [ ]:
query = "Sum the numbers 15 and 27 for me."
user_message = LLMMessage(
    role=Roles.USER, content=[MessageContent(type="text", text=query)]
)

# Define the LLM, but this time with the tool included. "auto" tool choice lets the LLM decide when to use the tool (not just always use it)
llm = OpenAILLM(tools=[tool_sum], tool_choice="auto")

response = await llm.generate([user_message])

print("\nRole: ", response.role)
print("Content: ", response.content[0].text if response.content else "(none)")
print("Tool Calls:")
for tool_call in response.tool_calls or []:
    print(f"  - {tool_call.function.name}({tool_call.function.arguments})")

## Run the Required Tools

Now that the LLM is able to ask to call the tool, we have to run the tools. 

Note: This step will typically be hidden by the next wrappers we will implement.

In [ ]:
results = {}

# For each tool the LLM asks for
for tool_call in response.tool_calls or []:
    tool_name = tool_call.function.name
    tool_args = tool_call.function.arguments

    # Find the tool by name
    tool = next((t for t in (llm.tools or []) if t.name == tool_name), None)

    # If the tool is found, call it with the provided arguments
    if tool is not None:
        tools_args_json = json.loads(tool_args)
        tool_result = await tool.call(tools_args_json)

        # Save the result of the tool call
        results[tool_name] = tool_result

for tool_name, result in results.items():
    print(f"Result from tool '{tool_name}': {result}")

## Inform the LLM of the outcome

Finally the LLM can get back the outcome of the prediction. We are simply sending back text to it.

In [ ]:
tools_answers = []

# For each tool result, create a LLMMessage to pass back to the LLM
for tool_call in response.tool_calls or []:
    tool_name = tool_call.function.name
    result = results[tool_name]
    call_id = tool_call.id

    tool_answer = LLMMessage(
        role=Roles.TOOL,
        name=tool_name,
        content=[MessageContent(text=json.dumps(result), type="text")],
        tool_call_id=call_id,
    )
    tools_answers.append(tool_answer)

In [ ]:
# Create the full conversation, including user message, LLM response, and tool answers
conversation = [user_message, response, *tools_answers]

In [ ]:
# The LLM can now generate a final response based on the conversation which includes the tool results
final_response = await llm.generate(conversation)

print("Role: ", final_response.role)
print("Content: ", final_response.content)

----------------------